# ResNet20 - Turntable Classification

**Model:** ResNet20  
**Dataset:** Turntable (12 classes, 96x96)  
**Loss:** sparse_categorical_crossentropy  
**Epochs:** 40 | **Batch Size:** 32 | **Optimizer:** Adam (lr=0.001)

In [ ]:
import numpy as np
import h5py
import os
import time
import csv

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import keras
from keras.layers import (Conv2D, BatchNormalization, Activation, Dense,
                          AveragePooling2D, Input, Flatten, Add)
from keras.models import Model
from keras.regularizers import l2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

OUTPUT_DIR = '.'
OUTPUT_DIR = '.'

In [ ]:
# --- Load Data ---
print("Loading turntable data...")
DATA_DIR = '../data_down'
    x_train, y_train = f['x_train'][:], f['y_train'][:]
    x_test, y_test = f['x_test'][:], f['y_test'][:]
    class_names = [c.decode() if isinstance(c, bytes) else str(c) for c in f['class_names'][:]]

print(f"Train: {x_train.shape}, Test: {x_test.shape}, Classes: {len(class_names)}")

In [ ]:
# --- Model Definition ---
def resnet_layer(inputs, num_filters=16, kernel_size=3, strides=1,
                 activation='relu', batch_normalization=True, conv_first=True):
    conv = Conv2D(num_filters, kernel_size=kernel_size, strides=strides,
                  padding='same', kernel_initializer='he_normal', kernel_regularizer=l2(1e-4))
    x = inputs
    if conv_first:
        x = conv(x)
        if batch_normalization: x = BatchNormalization()(x)
        if activation: x = Activation(activation)(x)
    else:
        if batch_normalization: x = BatchNormalization()(x)
        if activation: x = Activation(activation)(x)
        x = conv(x)
    return x

def resnet20_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    num_filters = 16
    x = resnet_layer(inputs=inputs, num_filters=num_filters)
    for stack in range(3):
        for block in range(3):
            strides = 1
            if stack > 0 and block == 0: strides = 2
            y = resnet_layer(inputs=x, num_filters=num_filters, strides=strides)
            y = resnet_layer(inputs=y, num_filters=num_filters, activation=None)
            if stack > 0 and block == 0:
                x = resnet_layer(inputs=x, num_filters=num_filters, kernel_size=1,
                                 strides=strides, activation=None, batch_normalization=False)
            x = Add()([x, y])
            x = Activation('relu')(x)
        num_filters *= 2
    x = AveragePooling2D(pool_size=8)(x)
    x = Flatten()(x)
    outputs = Dense(num_classes, activation='softmax', kernel_initializer='he_normal')(x)
    return Model(inputs=inputs, outputs=outputs, name='ResNet20')

model = resnet20_model((96,96,1), len(class_names))
print(f"Parameters: {model.count_params():,}")

In [ ]:
# --- Class weights ---
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(weights))

In [ ]:
# --- Train ---
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nTraining ResNet20 on Turntable (40 epochs)...")
start = time.time()
history = model.fit(x_train, y_train, epochs=40, batch_size=32,
                    validation_data=(x_test, y_test), class_weight=class_weight, verbose=2)
elapsed = time.time() - start
print(f"Training time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
# --- Evaluate ---
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {acc*100:.2f}%")
report = classification_report(y_test, y_pred, target_names=class_names, digits=3, zero_division=0)
print(report)

In [ ]:
# --- Save Model ---
OUTPUT_DIR = '.'

In [ ]:
# --- Save Single results.csv ---
report_dict = classification_report(y_test, y_pred, target_names=class_names,
                                     digits=4, zero_division=0, output_dict=True)
cm = confusion_matrix(y_test, y_pred)

OUTPUT_DIR = '.'
    w = csv.writer(f)

    w.writerow(['## Summary'])
    w.writerow(['metric', 'value'])
    w.writerow(['model', 'ResNet20'])
    w.writerow(['dataset', 'turntable'])
    w.writerow(['num_classes', len(class_names)])
    w.writerow(['input_shape', '96x96x1'])
    w.writerow(['parameters', model.count_params()])
    w.writerow(['epochs', 40])
    w.writerow(['batch_size', 32])
    w.writerow(['loss_function', 'sparse_categorical_crossentropy'])
    w.writerow(['optimizer', 'Adam (lr=0.001)'])
    w.writerow(['class_weights', 'balanced'])
    w.writerow(['test_accuracy', round(float(acc), 4)])
    w.writerow(['paper_accuracy', 0.936])
    w.writerow(['training_time_sec', round(elapsed, 1)])
    w.writerow([])

    w.writerow(['## Training History'])
    w.writerow(['epoch', 'loss', 'accuracy', 'val_loss', 'val_accuracy'])
    for i in range(len(history.history['loss'])):
        w.writerow([i+1,
                     round(history.history['loss'][i], 4),
                     round(history.history['accuracy'][i], 4),
                     round(history.history['val_loss'][i], 4),
                     round(history.history['val_accuracy'][i], 4)])
    w.writerow([])

    w.writerow(['## Per-Class Metrics'])
    w.writerow(['class', 'precision', 'recall', 'f1_score', 'support'])
    for cls in class_names:
        m = report_dict[cls]
        w.writerow([cls, round(m['precision'],4), round(m['recall'],4),
                     round(m['f1-score'],4), int(m['support'])])
    for avg in ['macro avg', 'weighted avg']:
        m = report_dict[avg]
        w.writerow([avg, round(m['precision'],4), round(m['recall'],4),
                     round(m['f1-score'],4), int(m['support'])])
    w.writerow([])

    w.writerow(['## Confusion Matrix'])
    w.writerow([''] + class_names)
    for i, cls in enumerate(class_names):
        w.writerow([cls] + cm[i].tolist())

print(f"\nSaved: model.keras + results.csv to {OUTPUT_DIR}")